In [1]:
from microtorch.model_maker import ModelMaker
from microtool.microtorch import MicrotorchTissueModel
from microtool.tissue_model import MultiTissueModel, RelaxationTissueModel
from microtool.utils.unit_registry import unit

microtorch_model = ModelMaker("Ball")

sphere = ModelMaker("Sphere")
sphere.fixed_D = 2

sphere_model = MicrotorchTissueModel(
    sphere,
    parameters={"radius": 6.0},
)

# Wrap each compartment individually, not the combined ModelMaker model.

ball_model = MicrotorchTissueModel(
    microtorch_model,
    parameters={"D": 2},
)



ball_relax = RelaxationTissueModel(ball_model, t2=250e-3)
sphere_relax = RelaxationTissueModel(sphere_model, t2 = 50e-3)


model = MultiTissueModel(
    [sphere_relax, ball_relax],
    volume_fractions=[0.6, 0.4],
)

model["model_0_T2"].scale = 1
model["model_1_T2"].scale = 1

model["vf_0"].fit_flag = True

model["model_1_model_D"].optimize = False
model["model_1_model_D"].fit_flag = False

model["vf_1"].optimize = False
model["vf_1"].fit_flag = False

model["S0"].optimize = True
model["S0"].fit_flag = True

print(model)

No YAML configuration found for Ball model.
Falling back to parsing model name for compartments: ['Ball'].
Parameter ranges will be the default compartment values.
-----------
########### Making model:  Ball
########### Compartments: ['Ball']
########### Parameter names: [['D']]
########### Parameter ranges: [[[0.001, 3]]]
-----------
No YAML configuration found for Sphere model.
Falling back to parsing model name for compartments: ['Sphere'].
Parameter ranges will be the default compartment values.
-----------
########### Making model:  Sphere
########### Compartments: ['Sphere']
########### Parameter names: [['radius']]
########### Parameter ranges: [[[0.001, 15]]]
-----------
Tissue model with 7 scalar parameters:
Tissue-parameter        Value    Scale  Optimize    Fit    Fit Bounds
--------------------  -------  -------  ----------  -----  -------------
model_0_model_radius     6           1  True        True   (0.001, 15.0)
model_0_T2               0.05        1  True        True 

In [2]:
import numpy as np
from microtool.acquisition_scheme import DiffusionAcquisitionScheme
from microtool.gradient_sampling import sample_uniform_half_sphere

n_directions = 6
n_shells = 6
block_size = n_directions + 1  # b0 + n_directions

# shell_bvals = [1500, 1000, 2200, 2500, 2500]
# shell_deltas = [0.0258, 0.0205,0.0205,0.0334, 0.0334]
# shell_Deltas = [0.0434, 0.0431,0.0431, 0.0455, 0.0455]
# initial_te = 0.09


shell_bvals = [2999, 2000, 1500, 500, 90, 150]
shell_deltas = [0.0247, 0.0132, 0.0247, 0.0122, 0.0122, 0.011]
shell_Deltas = [0.0438, 0.0323, 0.0434, 0.0313, 0.0238, 0.020]
initial_te = [0.090, 0.067, 0.090, 0.065, 0.050, 0.040]
# Build the measurements in shell blocks: b0, then six directions.
b_vals_list = []
pulse_durations_list = []
pulse_intervals_list = []
echo_times_list = []

for i in range(n_shells):
    b_vals_list.append(0.0)
    b_vals_list.extend([shell_bvals[i]] * n_directions)

    pulse_durations_list.extend([shell_deltas[i]] * block_size)
    pulse_intervals_list.extend([shell_Deltas[i]] * block_size)
    echo_times_list.extend([initial_te[i]] * block_size)

b_vals = np.array(b_vals_list)
pulse_durations = np.array(pulse_durations_list)
pulse_intervals = np.array(pulse_intervals_list)
echo_times = np.array(echo_times_list)
directions = sample_uniform_half_sphere(len(b_vals))

initial_scheme = DiffusionAcquisitionScheme.from_bvals(
    b_values=b_vals,
    b_vectors=directions,
    pulse_durations=pulse_durations,
    pulse_intervals=pulse_intervals,
    echo_times=echo_times,
)

# One optimized TE, delta, and Delta per shell, shared across that shell's
# b0 and diffusion-weighted measurements.
for name in (
    "EchoTime",
    "DiffusionPulseDuration",
    "DiffusionPulseInterval",
):
    initial_scheme[name].set_repetition_period(block_size)

# Keep each b0 gradient magnitude fixed at zero.
# All diffusion-weighted gradient magnitudes remain independently optimizable.
b0_mask = np.isclose(b_vals, 0.0)
magnitude = initial_scheme["DiffusionPulseMagnitude"]
magnitude.set_fixed_mask(b0_mask)

magnitude.set_fixed_mask(b0_mask)
magnitude.set_repetition_period(n_directions)

# Check the masks and b0 values.
print("Timing free indices:",
      np.flatnonzero(initial_scheme["EchoTime"].optimize_mask))
print("Gradient-magnitude free indices:",
      np.flatnonzero(magnitude.optimize_mask))
print("b0 gradient magnitudes:", magnitude.values[b0_mask])

assert np.all(magnitude.values[b0_mask] == 0.0)
assert not np.any(magnitude.optimize_mask[b0_mask])

print(magnitude.optimize_mask)
print(initial_scheme)


Timing free indices: [ 0  7 14 21 28 35]
Gradient-magnitude free indices: [ 1  8 15 22 29 36]
b0 gradient magnitudes: [-0. -0. -0. -0. -0. -0.]
[False  True False False False False False False  True False False False
 False False False  True False False False False False False  True False
 False False False False False  True False False False False False False
  True False False False False False]
Acquisition scheme with 42 measurements and 6 scalar parameters. 
total number of optimized parameters is 24:
  DiffusionPulseMagnitude [mT/mm] in (0.0, 0.078)    DiffusionGradientAnglePhi [rad] (fixed)    DiffusionGradientAngleTheta [rad] (fixed)    DiffusionPulseDuration [s] in (0.01, 0.1)    DiffusionPulseInterval [s] in (0.01, 0.1)    EchoTime [s] in (0.01, 0.1)
-------------------------------------------------  -----------------------------------------  -------------------------------------------  -------------------------------------------  -------------------------------------------  -

In [3]:
jacobian = model.scaled_jacobian(initial_scheme)
print(jacobian)

[[ 0.00000000e+00  3.57045599e+00  4.01861564e-01 -5.32377438e-01
   3.78249863e-01]
 [-8.44032406e-03  3.01772749e+00  9.98109342e-04  1.37976777e-01
   8.45188950e-02]
 [-8.44032406e-03  3.01772749e+00  9.98109342e-04  1.37976777e-01
   8.45188950e-02]
 [-8.44032406e-03  3.01772749e+00  9.98109342e-04  1.37976777e-01
   8.45188950e-02]
 [-8.44032406e-03  3.01772749e+00  9.98109342e-04  1.37976777e-01
   8.45188950e-02]
 [-8.44032406e-03  3.01772749e+00  9.98109342e-04  1.37976777e-01
   8.45188950e-02]
 [-8.44032406e-03  3.01772749e+00  9.98109342e-04  1.37976777e-01
   8.45188950e-02]
 [ 0.00000000e+00  4.21047835e+00  3.27992457e-01 -5.03062113e-01
   4.63070514e-01]
 [-1.51750316e-02  3.36436354e+00  6.00739154e-03  1.95216813e-01
   1.31139863e-01]
 [-1.51750316e-02  3.36436354e+00  6.00739154e-03  1.95216813e-01
   1.31139863e-01]
 [-1.51750316e-02  3.36436354e+00  6.00739154e-03  1.95216813e-01
   1.31139863e-01]
 [-1.51750316e-02  3.36436354e+00  6.00739154e-03  1.95216813e-01

/nfs/rtfs03/storage/home/jgroot34/miniforge3/envs/MICROtool/lib/python3.11/site-packages/torch/autograd/graph.py:1113: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


In [4]:
import copy
import numpy as np
import microtool.optimize.methods as methods
from microtool.optimize import optimize_scheme
from microtool.optimize.loss_functions import compute_loss, default_loss

n_runs = 1
noise_variance = 0.02
loss_function = default_loss  # or whichever loss function you intend to evaluate

optimization_history = []

for run_idx in range(n_runs):
    print(f"--- Running optimization {run_idx + 1}/{n_runs} ---")
    
    # Instantiate method per run to ensure clean state
    method = methods.SOMA(initial_search_attempts=500000)
    
    # Pass a deep copy of initial_scheme so in-place edits don't contaminate subsequent runs
    scheme_run, opt_info = optimize_scheme(
        copy.deepcopy(initial_scheme),
        model,
        noise_variance=noise_variance,
        method=method,
    )
    
    # Compute the final loss value for this optimized scheme
    current_loss = compute_loss(
        scheme_run,
        model,
        noise_variance,
        loss_function,
    )
    
    optimization_history.append({
        "run": run_idx + 1,
        "scheme": scheme_run,
        "loss": float(current_loss),
        "info": opt_info,
    })
    
    print(f"Run {run_idx + 1} completed with loss: {current_loss:.6e}")

# Find the run with the minimum loss
best_result = min(optimization_history, key=lambda res: res["loss"])
optimal_scheme_SOMA = best_result["scheme"]

print("\n=== Optimization Summary ===")
for res in optimization_history:
    print(f"Run {res['run']}: Loss = {res['loss']:.6e}")

print(f"\nBest Run: #{best_result['run']} with Lowest Loss = {best_result['loss']:.6e}")

--- Running optimization 1/1 ---
[SOMA] Starting Optimization
  Dimension (Nx)     : 24
  Population Size    : 50
  Max Migrations     : 200
  Max Fevals         : 240000
  Jumps per movement : 28


ValueError: Could not find a feasible, well-conditioned SOMA starting point after 500000 local perturbations from the supplied initial scheme. Provide a better starting scheme or increase initial_search_attempts.